<a href="https://colab.research.google.com/github/vraja10/500-AI-Agents-Projects/blob/main/Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Step 1: Install the Necessary Libraries

Before we start fine-tuning a language model, we need a few Python libraries that provide the tools required for the process.

Think of these libraries as **different tools in a toolbox**. Each one has a specific job.

### 🧰 What does each library do?

| Library | What we use it for |
|---|---|
| **`transformers`** | Load and work with pre-trained language models and tokenizers |
| **`peft`** | Apply parameter-efficient fine-tuning methods such as LoRA |
| **`accelerate`** | Helps run training efficiently on available hardware such as GPUs |
| **`bitsandbytes`** | Allows us to work with quantized models such as 4-bit models |
| **`trl`** | Provides tools for training language models, including `SFTTrainer` |
| **`torch`** | PyTorch, the deep-learning framework used by the model |
| **`pandas`** | Useful for working with structured data |
| **`scikit-learn`** | Provides machine-learning utilities |
| **`scipy`** | Provides scientific computing utilities |

### 🧠 The big picture

We will use these libraries to go through this pipeline:

```text
Load a pre-trained model
        ↓
Prepare it for efficient training
        ↓
Add LoRA
        ↓
Prepare a dataset
        ↓
Train the model
        ↓
Test the result
        ↓
Save the trained adapter

In [ ]:
# Install the required libraries
!pip install transformers peft accelerate bitsandbytes trl scipy pandas scikit-learn torch

# Restart the Colab runtime to ensure all new packages are loaded correctly.
# You will need to run this cell, then click 'RESTART RUNTIME' in the output, and then continue from the next cell.

# ⚙️ Step 2: Import Libraries and Set Up the Environment

Now that the required libraries are installed, we import the tools needed for the fine-tuning workflow.

### 📚 Libraries used

- **PyTorch (`torch`)** – The deep-learning framework used for tensors, model computation, and GPU/CPU handling.
- **Transformers** – Provides the pre-trained language model and tokenizer. The tokenizer converts text into tokens that the model can understand.
- **PEFT** – Provides **Parameter-Efficient Fine-Tuning** techniques such as LoRA. Instead of updating the entire model, we will train a small set of additional parameters.
- **BitsAndBytes** – Helps load the model using low-bit quantization such as **4-bit**, reducing GPU memory usage.
- **TRL** – Provides training utilities such as `SFTTrainer`, which we will use later for supervised fine-tuning.

### 🖥️ Selecting the computing device

The code also checks whether a **CUDA-compatible GPU** is available.

```text
CUDA GPU available?
       │
   ┌───┴───┐
   │       │
  YES      NO
   │       │
  GPU      CPU

In [ ]:
# Import necessary modules
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset

# Check if CUDA (NVIDIA GPU) is available and set the device
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

# Configure logging (optional, but good practice)
import logging
logging.basicConfig(level=logging.INFO)

GPU is available: Tesla T4


# 🤖 Step 3: Load a Pre-trained Model and Tokenizer

## 🎯 What are we doing?

Before we can fine-tune a language model, we first need to **load a pre-trained model**.

Training a large language model from scratch would require enormous amounts of data, computing power, and time. Instead, we start with a model that has already learned general language patterns and then adapt it to our task.

In this notebook, we use:

```text
TinyLlama/TinyLlama-1.1B-Chat-v1.0

In [ ]:
# Authenticate with Hugging Face if needed (for models like Llama 2)
# from huggingface_hub import notebook_login
# notebook_login()

# Define the base model ID
# Changed to an openly accessible model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Configure 4-bit quantization
bitsandbytes_config = BitsAndBytesConfig(
    load_in_4bit=True, # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4", # Use NF4 quantization type (often recommended)
    bnb_4bit_compute_dtype=torch.bfloat16, # Use bfloat16 for computation, which is faster on modern GPUs
    bnb_4bit_use_double_quant=False, # Disable double quantization for now (can save a little more memory)
)

# Load the base model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bitsandbytes_config,
    device_map={" ": 0} # Load the model onto the first GPU if available
)

# Disable cache for training to save memory
model.config.use_cache = False

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Set the padding token. Some models don't have a default pad token.
# We need a padding token for batching sequences of different lengths.
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Pad on the right side of the sequences

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# 🛠️ Step 4: Prepare the Model for K-bit Training

## 🎯 What are we doing?

In Step 3, we loaded TinyLlama using **4-bit quantization**.

Now we need to prepare this quantized model so that it can work correctly with **LoRA fine-tuning**.

But first, let's understand **why we are using k-bit quantization**.

---

## 🧠 Why do we use k-bit quantization?

A language model contains a huge number of parameters.

Each parameter is stored as a numerical value, and the number of bits used to represent those values affects how much memory the model requires.

For example:

```text
32-bit
████████████████████
More memory

16-bit
██████████
Less memory

4-bit
██
Much less memory

In [ ]:
# Prepare the model for k-bit training
model = prepare_model_for_kbit_training(model)
print("Model prepared for k-bit training.")

Model prepared for k-bit training.


# 🧩 Step 5: Configure LoRA for Parameter-Efficient Fine-Tuning

## 🎯 What are we doing?

In Step 4, we prepared our 4-bit model for training.

Now we need to decide **which part of the model we are actually going to train**.

Instead of updating all the parameters of the 1.1B-parameter TinyLlama model, we will use **LoRA**.

---

## 🧠 What is LoRA?

**LoRA** stands for **Low-Rank Adaptation**.

LoRA is a **Parameter-Efficient Fine-Tuning (PEFT)** technique that allows us to adapt a large pre-trained model by adding a small number of trainable parameters instead of updating the entire model.

In simple terms:

> **LoRA freezes the original model and adds small trainable matrices (adapters) to selected layers. During fine-tuning, these small adapters learn the new task while the original model remains largely unchanged.**

Without LoRA:

```text
Pre-trained Model
████████████████████████████
        ↓
Train all parameters
        ↓
Huge memory + computation requirement

In [ ]:
# Configure LoRA parameters
lora_config = LoraConfig(
    r=16, # LoRA attention dimension (rank)
    lora_alpha=32, # Scaling factor for LoRA updates
    target_modules=["q_proj", "v_proj"], # Apply LoRA to query and value projection layers
    lora_dropout=0.05, # Dropout probability for LoRA layers
    bias="none", # Do not train bias parameters
    task_type="CAUSAL_LM" # Task type is Causal Language Modeling
)

# Get the PEFT model by applying the LoRA configuration to the base model
model = get_peft_model(model, lora_config)

# Print the trainable parameters to see the significant reduction
model.print_trainable_parameters()
print("LoRA configuration applied to the model.")

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044
LoRA configuration applied to the model.


# 📚 Step 6: Load and Prepare the Dataset

## 🎯 What are we doing?

So far, we have prepared the model and added LoRA adapters.

Now we need to give the model **training data**.

A language model learns during fine-tuning by seeing examples of text and adjusting its trainable parameters based on those examples.

In this notebook, we use the:

```text
Abirate/english_quotes

In [ ]:
# Load a sample dataset for instruction fine-tuning
# You can replace 'timdettmers/openassistant-guanaco' with your own dataset.
dataset = load_dataset("Abirate/english_quotes")

# Let's inspect the dataset structure
print("Dataset structure:", dataset)
print("First example:", dataset['train'][0])

# Prepare the dataset for training by formatting the text
# This function will create a 'text' column that the SFTTrainer expects.
def format_instruction(example):
    # Adjust this function based on your dataset's specific column names
    # For 'Abirate/english_quotes', it has 'quote' and 'author' columns
    # We'll create a simple instruction: 'Quote by {author}: {quote}'
    return {"text": f"Quote by {example['author']}: {example['quote']}"}

# Apply the formatting function to the dataset
dataset = dataset.map(format_instruction, remove_columns=list(dataset['train'].features))

# Print a formatted example to verify
print("\nFormatted first example:", dataset['train'][0])

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})
First example: {'quote': '“Be yourself; everyone else is already taken.”', 'author': 'Oscar Wilde', 'tags': ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']}

Formatted first example: {'text': 'Quote by Oscar Wilde: “Be yourself; everyone else is already taken.”'}


# 🧪 Step 7: Test the Model Before Training the LoRA Adapter

## 🎯 What are we doing?

Before we actually train the model, we want to see **what it produces before training**.

This gives us a **baseline**.

Later, after training, we will give the model the same prompt again and compare the result.

```text
Before Training
      ↓
Generate a response
      ↓
Save/observe the output
      ↓
Train LoRA
      ↓
Generate again
      ↓
Compare Before vs After

@eval_prompt => you can change here and check output


In [ ]:
eval_prompt = "Quote by William Shakespeare: "

# Tokenize the input prompt
model_input = tokenizer(eval_prompt, return_tensors="pt").to(device)

# Generate a response from the model
model.eval() # Set model to evaluation mode
with torch.no_grad(): # Disable gradient calculations for inference
    response_tokens = model.generate(**model_input, max_new_tokens=50, do_sample=True, top_k=50, top_p=0.95, num_return_sequences=1)

# Decode and print the generated response
print("\n--- Model response BEFORE fine-tuning ---")
print(tokenizer.decode(response_tokens[0], skip_special_tokens=True))

# Set model back to training mode
model.train()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Model response BEFORE fine-tuning ---
Quote by William Shakespeare:  “When the dead are gathered, must not the living be told?”


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

# ⚙️ Step 8: Define the Training Arguments

## 🎯 What are we doing?

Now that the model, LoRA adapters, and dataset are ready, we need to define **how the model should be trained**.

For this, we use Hugging Face's `TrainingArguments`.

Think of `TrainingArguments` as the **control panel of the training process**. It defines settings such as:

- How long to train
- How many examples to process at once
- How quickly the model should learn
- Which optimizer to use
- How often to save and log the training progress

```text
Model + Dataset
      ↓
TrainingArguments
      ↓
Define HOW training should happen
      ↓
Trainer
      ↓
Fine-tuning

In [ ]:
from transformers import TrainingArguments

training_arguments = TrainingArguments(
    output_dir="./results", # Directory to save checkpoints and logs
    num_train_epochs=3, # Number of training epochs (adjust based on dataset size and desired performance)
    per_device_train_batch_size=4, # Batch size per GPU (adjust based on GPU memory)
    gradient_accumulation_steps=1, # Number of updates steps to accumulate before backward pass
    optim="paged_adamw_8bit", # Optimizer (memory-efficient version of AdamW)
    save_steps=100, # Save checkpoint every 100 steps
    logging_steps=100, # Log metrics every 100 steps
    learning_rate=2e-4, # Learning rate
    # Removed weight_decay as it caused TypeError or may be incompatible with older transformers versions
    # Removed warmup_ratio as it caused TypeError or may be incompatible with older transformers versions
    fp16=False, # Disable fp16 (use bf16 if available and supported by GPU)
    bf16=True, # Enable bf16 if supported, often more stable than fp16
    max_grad_norm=0.3, # Max gradient norm for clipping
    max_steps=-1, # -1 means train for num_train_epochs
    # Removed group_by_length as it caused TypeError or may be incompatible with older transformers versions
    lr_scheduler_type="constant", # Learning rate scheduler type
    report_to="tensorboard" # Report metrics to TensorBoard
)

print("Training arguments defined.")

Training arguments defined.


# 🏗️ Step 9: Set Up the SFTTrainer

## 🎯 What are we doing?

We have now prepared:

- The pre-trained TinyLlama model
- The 4-bit quantized model
- The LoRA adapters
- The training dataset
- The training configuration

Now we need to connect all of these components together.

For this, we use **`SFTTrainer`** from the TRL library.

**SFT** stands for **Supervised Fine-Tuning**.

The trainer is responsible for managing the actual training process, such as sending batches through the model, calculating the training loss, computing gradients, and updating the trainable LoRA parameters.

```text
Model
  +
Training Dataset
  +
Training Arguments
  ↓
SFTTrainer
  ↓
Ready to Train

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'], # Use the training split of the dataset
    # peft_config=lora_config, # Removed: model is already a PeftModel
    args=training_arguments
    # Removed max_seq_length as it caused TypeError for the current trl version
    # Removed packing as it caused TypeError for the current trl version
)

print("SFTTrainer initialized.")

SFTTrainer initialized.


# 🚀 Step 10: Start Fine-Tuning

## 🎯 What are we doing?

All the preparation is now complete.

We have:

```text
Pre-trained TinyLlama
        ↓
4-bit Quantization
        ↓
K-bit Training Preparation
        ↓
LoRA Adapters
        ↓
Training Dataset
        ↓
Training Arguments
        ↓
SFTTrainer

In [ ]:
# Start training
trainer.train()

print("Fine-tuning complete!")

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
100,2.164372
200,2.010282
300,1.885424
400,1.896062
500,1.955979
600,1.964042
700,1.886005
800,1.838341
900,1.805275
1000,1.832075


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss
100,2.164372
200,2.010282
300,1.885424
400,1.896062
500,1.955979
600,1.964042
700,1.886005
800,1.838341
900,1.805275
1000,1.832075


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Fine-tuning complete!


# 🧪 Step 11: Test the Model After Fine-Tuning

## 🎯 What are we doing?

In Step 7, we generated a response **before training** the LoRA adapters.

Now that training is complete, we generate another response using the fine-tuned model.

The purpose is to compare the model's behavior:

```text
Before Training
      ↓
Generate Response
      ↓
Train LoRA
      ↓
After Training
      ↓
Generate Response Again
      ↓
Compare the Outputs

In [ ]:
# Reload the tokenizer to ensure it's in a clean state
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Ensure the fine-tuned model is in evaluation mode
model.eval()

# Use the same prompt as before
# eval_prompt defined in a previous cell: "Quote by William Shake Sphere : "

# Tokenize the input prompt
model_input = tokenizer(eval_prompt, return_tensors="pt").to(device)

# Generate a response from the fine-tuned model
with torch.no_grad():
    response_tokens_after = model.generate(**model_input, max_new_tokens=50, do_sample=True, top_k=50, top_p=0.95, num_return_sequences=1)

# Decode and print the generated response
print("\n--- Model response AFTER fine-tuning ---")
print(tokenizer.decode(response_tokens_after[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Model response AFTER fine-tuning ---
Quote by William Shakespeare:  “What's past is past,What's gone is gone,And what's done is done:It's impossible to mend the damages which have been done,And when once it is done, It cannot be undone


# 💾 Step 12: Save the Fine-Tuned Model

## 🎯 What are we doing?

The training is now complete.

The LoRA adapters that were trained during Step 10 are currently available in our training environment.

If we close the Colab session without saving them, we could lose the trained result.

So in this final step, we save the fine-tuned model to a folder.

```text
Fine-Tuned Model
      ↓
Save LoRA Adapter
      ↓
./final_checkpoint/
      ↓
Can be reused later

In [ ]:
output_dir = "./final_checkpoint"
trainer.save_model(output_dir)

print(f"Fine-tuned model (LoRA adapters) saved to {output_dir}")
print("You can now download this folder and use the adapters with the base model.")

Fine-tuned model (LoRA adapters) saved to ./final_checkpoint
You can now download this folder and use the adapters with the base model.
